# Hotel Booking Cancellation Prediction – Stages 1, 2 and 3
**CDS 4003 – Final Project (Group 3)**

Stage 1: Business problem · Stage 2: Data collection · Stage 3: Data understanding and cleaning

---
# Part 1 – Business Problem
Predict at booking time whether a booking will be cancelled (`is_canceled`: 1 = yes, 0 = no) – binary classification.

**Success criteria:** F1 ≥ 0.70, ROC-AUC ≥ 0.85, recall (cancelled) ≥ 0.75 on the test set.

---
# Part 2 – Data Understanding
## 2.1 Load the data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREY = "#2b6cb0", "#dd6b20", "#a0aec0"
print("Libraries loaded")

No upload needed: the dataset is downloaded from the GitHub repository automatically.

In [ ]:
import os

# Where to find the dataset: local copy first, then Colab upload, then download from GitHub.
GITHUB_USER = "ixanm10"                          # change if the repo is under another account
GITHUB_REPO = "hotel-cancellation-prediction"    # change to your repository name
GITHUB_CSV  = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/main/data/hotel_bookings.csv"

candidates = ["data/hotel_bookings.csv", "../data/hotel_bookings.csv", "/content/hotel_bookings.csv"]
FILE_PATH = next((p for p in candidates if os.path.exists(p)), GITHUB_CSV)

raw = pd.read_csv(FILE_PATH)
print("Loaded:", FILE_PATH, "| Rows:", raw.shape[0], "| Columns:", raw.shape[1])
assert raw.shape[0] == 119390, "The dataset is incomplete - upload hotel_bookings.csv again and wait for it to finish"

## 2.2 Describe the data
119,390 rows × 32 columns (dataset approved by the instructor at Milestone 2).

In [ ]:
raw.head()

In [ ]:
raw.info()

### Column types
| Type | Columns |
|---|---|
| Nominal | hotel, meal, country, market_segment, distribution_channel, reserved_room_type, assigned_room_type, deposit_type, customer_type, reservation_status, agent, company |
| Binary | is_canceled (target), is_repeated_guest |
| Ordinal | arrival_date_month |
| Interval | arrival_date_year, arrival_date_week_number, arrival_date_day_of_month, reservation_status_date |
| Ratio | lead_time, stays_in_weekend_nights, stays_in_week_nights, adults, children, babies, previous_cancellations, previous_bookings_not_canceled, booking_changes, days_in_waiting_list, adr, required_car_parking_spaces, total_of_special_requests |

In [ ]:
nominal = raw.select_dtypes(exclude="number").columns.tolist()
numeric = raw.select_dtypes(include="number").columns.tolist()
print(len(nominal), "text/nominal columns:", nominal)
print()
print(len(numeric), "numeric columns:", numeric)

## 2.3 Descriptive statistics

In [ ]:
raw.describe().T.round(2)

In [ ]:
# Categorical columns: number of different values and the most frequent value (mode)
raw.describe(include="object").T

**Target distribution**

In [ ]:
print(raw["is_canceled"].value_counts())
print((raw["is_canceled"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

raw["is_canceled"].value_counts().plot(kind="bar", color=[BLUE, ORANGE])
plt.xticks([0, 1], ["Not cancelled (0)", "Cancelled (1)"], rotation=0)
plt.title("Label: is_canceled"); plt.ylabel("Number of bookings"); plt.show()

## 2.4 Data quality checks
Missing values, impossible values, inconsistent labels, duplicates, wrong data types.

In [ ]:
# 1. Missing values per column
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"missing": missing, "percent": (missing / len(raw) * 100).round(2)})

In [ ]:
# 2-5. Noise, inconsistencies, redundancy, incorrect types
guests_raw = raw["adults"] + raw["children"].fillna(0) + raw["babies"]
nights_raw = raw["stays_in_weekend_nights"] + raw["stays_in_week_nights"]

quality = {
    "1. Missing cells (total)":               int(raw.isna().sum().sum()),
    "2. Noise: bookings with 0 guests":       int((guests_raw == 0).sum()),
    "2. Noise: bookings with 0 nights":       int((nights_raw == 0).sum()),
    "2. Noise: negative price (adr < 0)":     int((raw["adr"] < 0).sum()),
    "2. Noise: extreme price (adr > 1000)":   int((raw["adr"] > 1000).sum()),
    "3. Inconsistent: meal = 'Undefined'":    int((raw["meal"] == "Undefined").sum()),
    "3. Inconsistent: segment = 'Undefined'": int((raw["market_segment"] == "Undefined").sum()),
    "3. Inconsistent: channel = 'Undefined'": int((raw["distribution_channel"] == "Undefined").sum()),
    "4. Redundancy: duplicate rows":          int(raw.duplicated().sum()),
    "5. Incorrect type: month stored as text":  raw["arrival_date_month"].dtype == object or str(raw["arrival_date_month"].dtype) == "str",
    "5. Incorrect type: date stored as text":   raw["reservation_status_date"].dtype == object or str(raw["reservation_status_date"].dtype) == "str",
    "5. Incorrect type: children is float":    str(raw["children"].dtype),
}
pd.Series(quality, name="result").to_frame()

In [ ]:
# Boxplots to SEE outliers in key numeric columns
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["lead_time", "adr", "stays_in_week_nights"]):
    sns.boxplot(x=raw[col], ax=ax, color=BLUE)
    ax.set_title(col)
plt.tight_layout(); plt.show()

**Summary:** missing values (mainly `company`, `agent`), ~32,000 duplicates, impossible bookings, "Undefined" labels and wrong data types.

---
# Part 3 – Data Preparation

In [ ]:
df = raw.copy()
log = []          # log of every preparation step
def record(step, action, reason, rows_affected=""):
    log.append({"step": step, "action": action, "reason": reason, "rows/cols affected": rows_affected})

## Step 1 – Select attributes
- `company` dropped (94% missing).
- `reservation_status`, `reservation_status_date`, `assigned_room_type` are kept for EDA but excluded from modelling (leakage: only known after booking).

In [ ]:
# Variance check: share of the most common value in each column
dominant = raw.apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0]).sort_values(ascending=False)
print("Share of the most frequent value (top 6):")
print((dominant.head(6) * 100).round(1).astype(str) + " %")

`babies` and `days_in_waiting_list` are mostly zero but not constant, so they are kept.

In [ ]:
df = df.drop(columns=["company"])
record(1, "Dropped column company", "94% missing values – almost empty", "1 column")
record(1, "Excluded reservation_status, reservation_status_date, assigned_room_type from modelling",
       "Data leakage – only known after the booking outcome", "3 columns")
print("Columns now:", df.shape[1])

## Step 2 – Clean data
### 2a. Missing values
| Column | Missing | Action |
|---|---|---|
| children | 4 | mode (0) |
| country | 488 | "Unknown" |
| agent | 16,340 | 0 (no agent) |

In [ ]:
print("children mode:", df["children"].mode()[0], "| median:", df["children"].median())

df["children"] = df["children"].fillna(df["children"].mode()[0])
df["country"]  = df["country"].fillna("Unknown")
df["agent"]    = df["agent"].fillna(0)

record(2, "children missing -> mode (0)", "Only 4 rows; most frequent value", "4 rows")
record(2, "country missing -> 'Unknown'", "Keeps the bookings; country not known", "488 rows")
record(2, "agent missing -> 0 ('no agent')", "Missing means booking without an agent", "16,340 rows")
print("Missing values left:", df.isna().sum().sum())

### 2b. Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
record(2, "Removed exact duplicate rows", "Redundant records bias the model and the test", f"{before - len(df):,} rows")
print("Duplicates removed:", before - len(df))

### 2c. Impossible values
Remove bookings with 0 guests, 0 nights, adr < 0 or adr > 1,000.

In [ ]:
before = len(df)
df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
df = df[(df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) > 0]
df = df[(df["adr"] >= 0) & (df["adr"] <= 1000)]
record(2, "Removed bookings with 0 guests, 0 nights, adr < 0 or adr > 1000", "Impossible values", f"{before - len(df):,} rows")
print("Invalid rows removed:", before - len(df))

### 2d. Outliers (IQR rule)
Detected but kept: long lead times and long stays are real behaviour.

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

pd.Series({c: iqr_outliers(df[c]) for c in ["lead_time", "adr", "stays_in_week_nights", "stays_in_weekend_nights"]},
          name="IQR outliers").to_frame()

In [ ]:
record(2, "Outliers detected with IQR rule but kept", "Long lead times / long stays are real behaviour and predictive", "0 rows")

## Step 3 – Format and standardize
meal "Undefined" → "SC"; remove "Undefined" segment/channel; month → number; dates → datetime; children/agent → int.

In [ ]:
df["meal"] = df["meal"].replace("Undefined", "SC")
before = len(df)
df = df[(df["market_segment"] != "Undefined") & (df["distribution_channel"] != "Undefined")]
record(3, "meal 'Undefined' -> 'SC'", "Same meaning per dataset documentation", "")
record(3, "Removed 'Undefined' market_segment / distribution_channel", "No meaning", f"{before - len(df)} rows")

month_map = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, "July": 7,
             "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}
df["arrival_month_num"] = df["arrival_date_month"].map(month_map)
df["arrival_date"] = pd.to_datetime(dict(year=df["arrival_date_year"], month=df["arrival_month_num"],
                                         day=df["arrival_date_day_of_month"]))
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])
df["children"] = df["children"].astype(int)
df["agent"] = df["agent"].astype(int)
record(3, "Month name -> number; dates -> datetime; children/agent -> int", "Correct types for calculations", "4 columns")

df = df.reset_index(drop=True)
df[["arrival_date_month", "arrival_month_num", "arrival_date", "children", "agent"]].dtypes

## Step 4 – New attributes
| Attribute | Formula |
|---|---|
| total_nights | weekend + week nights |
| total_guests | adults + children + babies |
| has_kids | children + babies > 0 |
| total_revenue | adr × total_nights |
| weekend_ratio | weekend nights / total nights |
| prev_cancel_rate | previous cancellations / all previous bookings |
| arrival_dow | weekday of arrival |
| is_summer | June–August |
| lead_time_band | lead time in 6 bins |
| country_grp | top 15 countries, rest "Other" |
| room_mismatch | reserved ≠ assigned room (EDA only) |

In [ ]:
df["total_nights"]  = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"]  = df["adults"] + df["children"] + df["babies"]
df["has_kids"]      = ((df["children"] + df["babies"]) > 0).astype(int)
df["total_revenue"] = df["adr"] * df["total_nights"]
df["weekend_ratio"] = df["stays_in_weekend_nights"] / df["total_nights"]

prev_total = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
df["prev_cancel_rate"] = (df["previous_cancellations"] / prev_total.replace(0, np.nan)).fillna(0)

df["arrival_dow"] = df["arrival_date"].dt.dayofweek          # 0 = Monday
df["is_summer"]   = df["arrival_month_num"].isin([6, 7, 8]).astype(int)

lead_bins   = [-1, 7, 30, 90, 180, 365, 800]
lead_labels = ["0-7", "8-30", "31-90", "91-180", "181-365", "365+"]
df["lead_time_band"] = pd.cut(df["lead_time"], lead_bins, labels=lead_labels)

top_countries = df["country"].value_counts().head(15).index
df["country_grp"] = np.where(df["country"].isin(top_countries), df["country"], "Other")

df["room_mismatch"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

record(4, "Created 11 new attributes (totals, ratios, binning, date features, grouping)", "Expose patterns to the model", "11 columns")
df[["total_nights", "total_guests", "total_revenue", "weekend_ratio", "prev_cancel_rate",
    "arrival_dow", "is_summer", "lead_time_band", "country_grp"]].head()

## Step 5 – Integrate datasets
Not needed: single table.

In [ ]:
record(5, "No integration needed", "All attributes are in a single file", "-")

## Step 6 – Encode and normalize
One-hot + StandardScaler (linear models), ordinal encoding (trees). Applied inside Pipelines during modelling, fitted on training data only. Example:

In [ ]:
pd.get_dummies(df[["deposit_type"]].head(6), dtype=int)

In [ ]:
record(6, "One-hot (linear models) / ordinal (trees) encoding + Z-normalization inside Pipelines",
       "Models need numeric input; fitted on training data only to avoid leakage", "")

## Step 7 – Preparation log and before/after table

In [ ]:
pd.DataFrame(log)

In [ ]:
before_after = pd.DataFrame({
    "Before cleaning": [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "After cleaning":  [df.shape[0],  df.shape[1],  int(df.isna().sum().sum()),  int(df.duplicated().sum())],
}, index=["Rows", "Columns", "Missing values", "Duplicate rows"])
before_after

In [ ]:
df.to_csv("hotel_bookings_clean.csv", index=False)
print("Clean data stored as hotel_bookings_clean.csv")

---
# End of Stages 1–3
Clean data saved as `hotel_bookings_clean.csv`.